# Forward solve: linear-solver timing comparison

This tutorial runs the `coil-fem` structural-analysis forward solve on a single
W7-X coil and compares how long `CoilFEM.run()` takes with three linear solvers:

- `umfpack` — CPU sparse direct (SuiteSparse), the default.
- `petsc` — PETSc iterative solver (jax_fem defaults).
- `cudss` — GPU sparse direct via spineax + NVIDIA cuDSS (needs the `coil-fem[cudss]` extra).

For each solver we time two things separately:

- **compile + run** — the first `run()` call, which includes JAX/XLA compilation.
- **warm run** — a second `run()` call, reusing the compiled program.

The geometry, mesh, material, and Winkler support setup are adapted from the
W7-X convergence benchmark, but the notebook is self-contained: the coil
geometry comes from `simsopt.configs.get_data('w7x')`.

In [1]:
import time

import jax
import jax.numpy as jnp

# cuDSS (and the FEM solve) run in double precision.
jax.config.update("jax_enable_x64", True)

from coil_fem.geo import CurveXYZFourierJAX
from coil_fem.container import CoilFEM

print("JAX devices:", jax.devices())

[06-21 15:36:05][INFO] jax_fem: pyamgx not installed. AMGX solver disabled.


       __       ___      ___   ___                _______  _______ .___  ___. 
      |  |     /   \     \  \ /  /               |   ____||   ____||   \/   | 
      |  |    /  ^  \     \  V  /      ______    |  |__   |  |__   |  \  /  | 
.--.  |  |   /  /_\  \     >   <      |______|   |   __|  |   __|  |  |\/|  | 
|  `--'  |  /  _____  \   /  .  \                |  |     |  |____ |  |  |  | 
 \______/  /__/     \__\ /__/ \__\               |__|     |_______||__|  |__| 
                                                                              



JAX devices: [CudaDevice(id=0)]


## Load the coil geometry

We use the first W7-X coil from simsopt and represent it as a JAX Fourier curve
sampled at 100 quadrature points along the centreline.

In [2]:
from simsopt.configs import get_data

curves, currents, axis, nfp_w7x, bs = get_data("w7x")

# Single coil (coil 0), no field-period symmetry.
n_quadpoints = 100
quadpoints = jnp.linspace(0, 1, n_quadpoints, endpoint=False)

base_curves_jax = [
    CurveXYZFourierJAX(
        quadpoints=quadpoints,
        dofs=curves[0].get_dofs(),
        order=curves[0].order,
    )
]
base_currents_jax = [currents[0].current]

print(f"coil 0: order={curves[0].order}, {n_quadpoints} quadrature points")

coil 0: order=48, 100 quadrature points


## Mesh, material, and support options

A rectangular cross-section is swept along the coil and meshed with TET10
elements. The coil is held by a soft-sphere Winkler support at its top and
bottom (highest/lowest points of the centreline); this removes rigid-body modes
so the problem is non-singular.

In [3]:
mesh_options = dict(
    shape="rect",
    w1=0.2,            # half-width [m]
    w2=0.2,            # half-width [m]
    frame="rmf",       # rotation-minimising frame
    aspect_ratio=1.0,  # aim for cubic elements
    mesh_type="TET10",
)

material_options = dict(
    E=200e9,        # Young's modulus [Pa]
    nu=0.30,        # Poisson ratio
    density=7800.0, # mass density [kg/m^3]
)

winkler_k = 1e10  # Winkler spring stiffness [N/m^3]

# Soft-sphere support clamping the coil top and bottom.
clamp_radius = 2 * max(mesh_options["w1"], mesh_options["w2"])
sigmoid_beta = 20.0 / clamp_radius


def support_fn(surface_points, coil, dofs):
    """Weights in [0, 1]: ~1 near the coil's top/bottom, ~0 elsewhere."""
    gamma = coil.gamma()
    top = gamma[jnp.argmax(gamma[:, 2])]
    bottom = gamma[jnp.argmin(gamma[:, 2])]

    # +1e-30 keeps the gradient of the norm finite at zero distance.
    d_top = jnp.sqrt(jnp.sum((surface_points - top) ** 2, axis=-1) + 1e-30)
    d_bottom = jnp.sqrt(jnp.sum((surface_points - bottom) ** 2, axis=-1) + 1e-30)

    w_top = jax.nn.sigmoid(sigmoid_beta * (clamp_radius - d_top))
    w_bottom = jax.nn.sigmoid(sigmoid_beta * (clamp_radius - d_bottom))
    return jnp.maximum(w_top, w_bottom)

## Time the forward solve per solver

The linear solver is fixed when the `CoilFEM` object is built (it wires up the
solver into the differentiable forward pass), so we build a fresh `CoilFEM` for
each solver. Construction happens outside the timed region; we only time the two
`run()` calls. `jax.clear_caches()` is called before each solver so the
compile time is measured honestly.

In [4]:
def build_fem(solver):
    return CoilFEM(
        base_curves_jax=base_curves_jax,
        base_currents_jax=base_currents_jax,
        base_support_fns=support_fn,
        base_support_dofs=None,
        nfp=1,
        stellsym=False,
        mesh_options=mesh_options,
        material_options=material_options,
        problem_options={
            "winkler_k": winkler_k,
            "solver": solver,
            "adjoint_solver": solver,
        },
    )


def time_run(fem):
    t0 = time.perf_counter()
    jax.block_until_ready(fem.run())            # compile + run
    t_jit = time.perf_counter() - t0

    t0 = time.perf_counter()
    jax.block_until_ready(fem.run())            # warm run (compiled)
    t_run = time.perf_counter() - t0
    return t_jit, t_run


timings = {}
for solver in ["umfpack", "petsc", "cudss"]:
    jax.clear_caches()
    try:
        timings[solver] = time_run(build_fem(solver))
        print(f"{solver:>8}: compile+run = {timings[solver][0]:.3f}s   warm = {timings[solver][1]:.3f}s")
    except Exception as e:
        timings[solver] = (None, None)
        print(f"{solver:>8}: FAILED -> {e}")

[06-21 15:36:07][DEBUG] jax_fem: Computing shape function values, gradients, etc.


[06-21 15:36:07][DEBUG] jax_fem: ele_type = TET10, quad_points.shape = (num_quads, dim) = (4, 3)


[06-21 15:36:07][DEBUG] jax_fem: face_quad_points.shape = (num_faces, num_face_quads, dim) = (4, 3, 3)


[06-21 15:36:07][DEBUG] jax_fem: Done pre-computations, took 0.0197446346282959 [s]


[06-21 15:36:07][INFO] jax_fem: Solving a problem with 5400 cells, 10400x3 = 31200 dofs.


...

[06-21 15:36:54][INFO] jax_fem: CuDSSNewtonSolver: building BC metadata …


[06-21 15:36:54][INFO] jax_fem: CuDSSNewtonSolver: instantiating cuDSS solver …


/home/lf2869/Documents/Codes/coil-fem/src/coil_fem/cudss_solver.py:384: UserWarning: A JAX array is being set as static! This can result in unexpected behavior and is usually a mistake to do.
  self.cudss = CuDSSSolver(


solving with float64


   cudss: compile+run = 8.447s   warm = 0.173s


In [5]:
print(f"{'solver':>8} | {'compile+run [s]':>16} | {'warm run [s]':>13}")
print("-" * 44)
for solver, (t_jit, t_run) in timings.items():
    jit_str = f"{t_jit:.3f}" if t_jit is not None else "n/a"
    run_str = f"{t_run:.3f}" if t_run is not None else "n/a"
    print(f"{solver:>8} | {jit_str:>16} | {run_str:>13}")

  solver |  compile+run [s] |  warm run [s]
--------------------------------------------
 umfpack |           16.100 |         8.166
   petsc |              n/a |           n/a
   cudss |            8.447 |         0.173


## Notes

- `CoilFEM.run()` is forward-only (no gradients); use `CoilFEM.objective` for optimisation.
- `umfpack` and `petsc` run on the CPU; `cudss` runs on the GPU. The warm-run
  time isolates the linear solve from compilation and is where the GPU direct
  solver typically differs most.
- `jax.clear_caches()` between solvers ensures the first `run()` really pays the
  compilation cost.
- `petsc` uses jax_fem's default iterative settings; if it fails to converge it
  is reported as `FAILED` and skipped without stopping the notebook.
- The `cudss` solver requires the optional GPU stack: `pip install --no-build-isolation -e ".[cudss]"`
  (see the project README / AGENTS.md for the `nvcc` prerequisite).